# Model results figures (from Excel logs)

Reads result workbooks under results/logs/ and writes per-family figure sets.


- ["random_forest"] → results/figures/rf_from_excel/
- ["gradboost"] → GB (+ MERF-GBR / FT-Transformer merge) → gradboost_from_excel/
- ["svm"] → SVM (+ MERF-SVR merge) → svm_from_excel/
- ["merf_gbr"] → merf_gbr_from_excel/
- ["gradboost", "random_forest", "svm", "merf_gbr"] — all

Cross-model comparison figures: run unified_leaderboard.py.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

BASE = Path(r"C:/Users/janku/Documents/KCL/Research Project/Research Project")

CONFIGS = {
  "gradboost": {
    "excel_path": BASE / "results" / "logs" / "GB Results" / "gradient_boosting_results.xlsx",
    "fig_dir": BASE / "results" / "figures" / "gradboost_from_excel",
    "merge_supplementary": True,
    "title": "Gradient Boosting",
  },
  "random_forest": {
    "excel_path": BASE / "results" / "logs" / "RF Results" / "random_forest_results.xlsx",
    "fig_dir": BASE / "results" / "figures" / "rf_from_excel",
    "merge_supplementary": False,
    "title": "Random Forest",
  },
  "svm": {
    "excel_path": BASE / "results" / "logs" / "SVM Results" / "svm_results.xlsx",
    "fig_dir": BASE / "results" / "figures" / "svm_from_excel",
    "merge_supplementary": True,
    "title": "SVM / MERF-SVR",
  },
  "merf_gbr": {
    "excel_path": BASE / "results" / "logs" / "GB Results" / "merf_gbr_results.xlsx",
    "fig_dir": BASE / "results" / "figures" / "merf_gbr_from_excel",
    "merge_supplementary": False,
    "title": "MERF-GBR",
  },
}

# Set which result workbooks to plot (run all cells after changing this).
ACTIVE_RUNS = ["gradboost", "random_forest", "svm", "merf_gbr"]

SHEETS = [
  "split_info",
  "train_test_distribution",
  "cv_folds",
  "cv_balance",
  "cv_summary",
  "subgroup_gender",
  "subgroup_age",
  "subgroup_gender_age",
  "fairness",
  "test_summary",
]

MERF_GBR_EXCEL_PATH = BASE / "results" / "logs" / "GB Results" / "merf_gbr_results.xlsx"
MERF_SVR_EXCEL_PATH = BASE / "results" / "logs" / "SVM Results" / "merf_svr_results.xlsx"
FT_RADAR_METRICS_DIR = BASE / "results" / "metrics" / "ft_transformer" / "RADAR"
FT_TRANSFORMER_RESULTS_XLSX = FT_RADAR_METRICS_DIR / "ft_transformer_radar_test_results.xlsx"


def save_fig(fig, fig_dir, name, dpi=200):
  fig_dir = Path(fig_dir)
  fig_dir.mkdir(parents=True, exist_ok=True)
  path = fig_dir / name
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
  plt.close(fig)
  print(f"Saved: {path}")
  return path


def load_results(path):
  path = Path(path)
  if not path.exists():
    raise FileNotFoundError(f"Results file not found: {path}")
  data = {}
  xl = pd.ExcelFile(path)
  for sheet in SHEETS:
    if sheet in xl.sheet_names:
      data[sheet] = pd.read_excel(path, sheet_name=sheet)
    else:
      print(f"Warning: missing sheet '{sheet}' in {path.name}")
  return data


def _concat_dedupe_frames(base, extras, subset_keys):
  frames = []
  if base is not None and not base.empty:
    frames.append(base)
  for extra in extras:
    if extra is not None and not extra.empty:
      frames.append(extra)
  if not frames:
    return pd.DataFrame()
  out = pd.concat(frames, ignore_index=True)
  if subset_keys and all(k in out.columns for k in subset_keys):
    out = out.drop_duplicates(subset=list(subset_keys), keep="last")
  return out


def merge_supplementary_results(data, cfg):
  """Append supplementary workbooks based on config key."""
  run_key = cfg.get("run_key", "")
  merf_paths = []
  if run_key == "gradboost":
    merf_paths.append(Path(MERF_GBR_EXCEL_PATH))
  elif run_key == "svm":
    merf_paths.append(Path(MERF_SVR_EXCEL_PATH))

  ft_dir = Path(FT_RADAR_METRICS_DIR)

  def _extras_for_sheet(sheet):
    chunks = []
    for merf_path in merf_paths:
      if merf_path.exists():
        merf_sheets = set(pd.ExcelFile(merf_path).sheet_names)
        if sheet in merf_sheets:
          chunks.append(pd.read_excel(merf_path, sheet_name=sheet))
    if run_key == "gradboost":
      csv_p = ft_dir / f"{sheet}.csv"
      if csv_p.exists():
        chunks.append(pd.read_csv(csv_p))
    return chunks

  for sheet in ("subgroup_gender", "subgroup_age", "subgroup_gender_age", "fairness"):
    base = data.get(sheet)
    if base is None:
      base = pd.DataFrame()
    extras = _extras_for_sheet(sheet)
    subset = ["model"] if sheet == "fairness" else ["model", "group"]
    if not extras and base.empty:
      continue
    merged = _concat_dedupe_frames(base, extras, subset)
    if not merged.empty:
      data[sheet] = merged

  ts_base = data.get("test_summary")
  if ts_base is None:
    ts_base = pd.DataFrame()
  ts_extras = []
  for merf_path in merf_paths:
    if merf_path.exists() and "test_summary" in pd.ExcelFile(merf_path).sheet_names:
      ts_extras.append(pd.read_excel(merf_path, sheet_name="test_summary"))
  if run_key == "gradboost":
    ft_xlsx = Path(FT_TRANSFORMER_RESULTS_XLSX)
    if ft_xlsx.exists() and "test_summary" in pd.ExcelFile(ft_xlsx).sheet_names:
      ts_extras.append(pd.read_excel(ft_xlsx, sheet_name="test_summary"))
  merged_ts = _concat_dedupe_frames(ts_base, ts_extras, ["model"])
  if not merged_ts.empty:
    data["test_summary"] = merged_ts

  print(f"\n=== Supplementary merge ({run_key}) ===")
  for merf_path in merf_paths:
    print(f"  MERF workbook exists: {merf_path.exists()} — {merf_path}")
  if run_key == "gradboost":
    print(f"  FT-Transformer metrics dir: {ft_dir}")


def enrich_model_columns(df):
  out = df.copy()
  if "model" not in out.columns:
    return out
  out["dataset"] = np.where(
    out["model"].str.contains("RADAR", case=False, na=False),
    "RADAR",
    np.where(out["model"].str.contains("Android", case=False, na=False), "Androids", "Other"),
  )
  out["model_short"] = (
    out["model"].str.replace("RADAR ", "", regex=False).str.replace("Androids ", "", regex=False)
  )
  out["model_label"] = out["dataset"] + " | " + out["model_short"]
  return out


def numeric_metric_cols(df, exclude=None):
  exclude = set(exclude or [])
  skip = {"model", "dataset", "model_short", "model_label", "fold", "group", "label", "matrix", "subset", "comparison", "test_name"}
  skip |= exclude
  cols = []
  for c in df.columns:
    if c in skip:
      continue
    if pd.api.types.is_numeric_dtype(df[c]):
      cols.append(c)
  return cols


def normalize_gender_token(value):
  if pd.isna(value):
    return value
  token = str(value).strip().lower()
  mapping = {
    "gender_0": "female", "gender_1": "male", "0": "female", "1": "male",
    "f": "female", "m": "male", "female": "female", "male": "male",
    "woman": "female", "man": "male",
  }
  return mapping.get(token, str(value).strip())


def normalize_subgroup_group_value(value):
  if pd.isna(value):
    return value
  text = str(value).strip()
  parts = text.split()
  if len(parts) >= 2:
    age_part = parts[0]
    gender_part = " ".join(parts[1:])
    return f"{age_part} {normalize_gender_token(gender_part)}"
  return normalize_gender_token(text)


def combine_gender_subgroups(df, group_col="group"):
  out = df.copy()
  out[group_col] = out[group_col].map(normalize_subgroup_group_value)
  id_cols = [c for c in ["model", "dataset", "model_short", "model_label", group_col] if c in out.columns]
  agg_dict = {}
  for col in out.columns:
    if col in id_cols:
      continue
    if col in {"n", "n_depressed", "n_control"}:
      agg_dict[col] = "sum"
    elif col in {"accuracy", "f1", "roc_auc"}:
      agg_dict[col] = "mean"
  if not agg_dict:
    return out.drop_duplicates(subset=id_cols)
  return out.groupby(id_cols, as_index=False).agg(agg_dict)


def prepare_data(cfg):
  data = load_results(cfg["excel_path"])
  if cfg.get("merge_supplementary"):
    merge_supplementary_results(data, cfg)
  for k, v in list(data.items()):
    data[k] = enrich_model_columns(v)
    models = data[k]["model"].unique().tolist() if "model" in data[k].columns else "n/a"
    print(f"  {k}: {v.shape} — models: {models}")
  return data


def plot_subgroup_sheet(data, fig_dir, sheet_key, sheet_name, filename_prefix, normalize_gender=False):
  df = data.get(sheet_key)
  if df is None or df.empty:
    print(f"Skipping {sheet_name}: no data")
    return
  metrics = [c for c in ["accuracy", "f1", "roc_auc"] if c in df.columns]
  if not metrics:
    return
  group_col = "group" if "group" in df.columns else df.columns[2]
  if normalize_gender:
    df = combine_gender_subgroups(df, group_col)
  long = df.melt(
    id_vars=["model_label", "dataset", "model_short", group_col],
    value_vars=metrics,
    var_name="metric",
    value_name="score",
  )
  long = long.rename(columns={group_col: "subgroup"})
  subgroup_order = None
  if normalize_gender:
    present = df[group_col].dropna().unique().tolist()
    if all(" " not in str(g) for g in present):
      subgroup_order = [g for g in ["female", "male"] if g in present]
    else:
      age_order = ["young", "middle", "older"]
      gender_order = ["female", "male"]
      subgroup_order = [
        f"{age} {gender}" for age in age_order for gender in gender_order
        if f"{age} {gender}" in present
      ]
  for metric in metrics:
    sub = long[long["metric"] == metric]
    g = sns.catplot(
      data=sub, kind="bar", x="subgroup", y="score", hue="model_short",
      col="dataset", col_wrap=2, height=5, aspect=1.3, sharey=True, order=subgroup_order,
    )
    g.fig.suptitle(f"{sheet_name}: {metric} by subgroup", y=1.02)
    for ax in g.axes.ravel():
      ax.tick_params(axis="x", rotation=40)
    save_fig(g.fig, fig_dir, f"{filename_prefix}_{metric}.png")
  if "accuracy" in metrics:
    pivot = df.pivot_table(index="model_label", columns=group_col, values="accuracy", aggfunc="first")
    if normalize_gender and subgroup_order:
      cols = [c for c in subgroup_order if c in pivot.columns]
      pivot = pivot[cols]
    fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1] * 0.8), max(4, pivot.shape[0] * 0.45)))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"{sheet_name}: accuracy by model and subgroup")
    save_fig(fig, fig_dir, f"{filename_prefix}_accuracy_heatmap.png")


def generate_all_figures(cfg):
  fig_dir = Path(cfg["fig_dir"])
  fig_dir.mkdir(parents=True, exist_ok=True)
  print(f"\n{'=' * 60}\nGenerating figures: {cfg['title']}\n  Excel: {cfg['excel_path']}\n  Output: {fig_dir}\n{'=' * 60}")
  data = prepare_data(cfg)

  # 1. Split info
  if "split_info" in data:
    split_df = data["split_info"]
    melted = split_df.melt(
      id_vars=["model_label", "dataset", "model_short"],
      value_vars=["train_rows", "test_rows", "train_participants", "test_participants"],
      var_name="split_metric", value_name="count",
    )
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(data=melted, x="model_label", y="count", hue="split_metric", ax=ax)
    ax.set_title(f"{cfg['title']}: data split sizes by model")
    ax.set_xlabel("Model")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=35)
    plt.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_fig(fig, fig_dir, "01_split_info.png")

  # 2. Train/test distribution
  if "train_test_distribution" in data and not data["train_test_distribution"].empty:
    dist_df = data["train_test_distribution"]
    value_cols = [c for c in dist_df.columns if c.lower() in {"train", "test"}]
    if not value_cols:
      value_cols = [
        c for c in dist_df.columns
        if c not in {"model", "dataset", "model_short", "model_label", "label"}
        and pd.api.types.is_numeric_dtype(dist_df[c])
      ]
    melted = dist_df.melt(
      id_vars=["model_label", "dataset", "label"],
      value_vars=value_cols, var_name="split", value_name="count",
    )
    g = sns.catplot(
      data=melted, kind="bar", x="label", y="count", hue="split",
      col="dataset", col_wrap=2, height=4, aspect=1.2, sharey=False,
    )
    g.fig.suptitle(f"{cfg['title']}: train vs test class distribution", y=1.02)
    g.set_axis_labels("Class", "Count")
    save_fig(g.fig, fig_dir, "02_train_test_distribution_by_dataset.png")

  # 3. CV folds
  if "cv_folds" in data:
    cv_folds = data["cv_folds"].copy()
    if "fold" not in cv_folds.columns:
      cv_folds = cv_folds.reset_index()
    plot_metrics = [c for c in ["accuracy", "f1", "roc_auc", "mae", "rmse", "r2"] if c in cv_folds.columns and cv_folds[c].notna().any()]
    if plot_metrics:
      fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 5), squeeze=False)
      for ax, metric in zip(axes.ravel(), plot_metrics):
        sns.lineplot(data=cv_folds, x="fold", y=metric, hue="model_label", marker="o", ax=ax)
        ax.set_title(f"CV {metric} by fold")
        ax.set_xlabel("Fold")
        ax.set_xticks(sorted(cv_folds["fold"].dropna().unique()))
        ax.legend(title="", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
      fig.tight_layout()
      save_fig(fig, fig_dir, "03_cv_folds_lineplots.png")
      for metric in plot_metrics:
        pivot = cv_folds.pivot_table(index="model_short", columns="fold", values=metric, aggfunc="first")
        if pivot.empty:
          continue
        fig, ax = plt.subplots(figsize=(8, max(4, 0.5 * len(pivot))))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax)
        ax.set_title(f"CV {metric} — model x fold")
        save_fig(fig, fig_dir, f"03_cv_folds_heatmap_{metric}.png")

  # 4. CV balance
  if "cv_balance" in data:
    bal = data["cv_balance"].copy()
    if "fold" not in bal.columns and bal.index.name == "fold":
      bal = bal.reset_index()
    balance_metrics = numeric_metric_cols(bal, exclude={"n_rows"})
    if "n_rows" in bal.columns:
      balance_metrics = ["n_rows"] + [c for c in balance_metrics if c != "n_rows"]
    for metric in balance_metrics[:4]:
      fig, ax = plt.subplots(figsize=(12, 5))
      sns.barplot(data=bal, x="fold", y=metric, hue="model_label", ax=ax)
      ax.set_title(f"CV fold balance: {metric}")
      ax.set_xlabel("Fold")
      ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
      save_fig(fig, fig_dir, f"04_cv_balance_{metric}.png")

  # 5. CV summary
  if "cv_summary" in data:
    cv_sum = data["cv_summary"]
    mean_cols = [c for c in cv_sum.columns if c.endswith("_mean") and pd.api.types.is_numeric_dtype(cv_sum[c])]
    for mean_col in mean_cols:
      std_col = mean_col.replace("_mean", "_std")
      plot_df = cv_sum[["model_label", "dataset", mean_col]].copy()
      plot_df["value"] = plot_df[mean_col]
      plot_df["err"] = cv_sum[std_col] if std_col in cv_sum.columns else 0
      fig, ax = plt.subplots(figsize=(12, 5))
      x = np.arange(len(plot_df))
      colors = sns.color_palette("Set2", len(plot_df))
      ax.bar(x, plot_df["value"], yerr=plot_df["err"], capsize=4, color=colors)
      ax.set_xticks(x)
      ax.set_xticklabels(plot_df["model_label"], rotation=35, ha="right")
      ax.set_title(f"CV summary: {mean_col}")
      ax.set_ylabel(mean_col)
      save_fig(fig, fig_dir, f"05_cv_summary_{mean_col}.png")
    if mean_cols:
      long = cv_sum.melt(
        id_vars=["model_label", "dataset", "model_short"],
        value_vars=mean_cols, var_name="metric", value_name="value",
      )
      g = sns.catplot(
        data=long, kind="bar", x="model_short", y="value", hue="dataset",
        col="metric", col_wrap=3, height=4, aspect=1.1, sharey=False,
      )
      g.fig.suptitle("CV summary metrics by dataset and model", y=1.02)
      for ax in g.axes.ravel():
        ax.tick_params(axis="x", rotation=30)
      save_fig(g.fig, fig_dir, "05_cv_summary_all_metrics.png")

  # 6. Subgroups
  plot_subgroup_sheet(data, fig_dir, "subgroup_gender", "Gender subgroups", "06_subgroup_gender", normalize_gender=True)
  plot_subgroup_sheet(data, fig_dir, "subgroup_age", "Age subgroups", "07_subgroup_age")
  plot_subgroup_sheet(data, fig_dir, "subgroup_gender_age", "Gender x age subgroups", "08_subgroup_gender_age", normalize_gender=True)

  # 7. Fairness
  if "fairness" in data and not data["fairness"].empty:
    fair = data["fairness"]
    fair_metrics = [c for c in ["disparate_impact", "statistical_parity_difference"] if c in fair.columns]
    melted = fair.melt(
      id_vars=["model_label", "dataset", "model_short"],
      value_vars=fair_metrics, var_name="fairness_metric", value_name="value",
    )
    fig_w = max(12.0, 1.1 * melted["model_label"].nunique())
    fig, ax = plt.subplots(figsize=(fig_w, 5))
    sns.barplot(data=melted, x="model_label", y="value", hue="fairness_metric", ax=ax)
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="DI = 1 (parity)")
    ax.axhline(0.0, color="black", linestyle=":", linewidth=1)
    ax.set_title("Fairness metrics on held-out predictions")
    ax.tick_params(axis="x", rotation=35)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    save_fig(fig, fig_dir, "09_fairness_combined.png")
    g = sns.catplot(
      data=melted, kind="bar", x="model_short", y="value", hue="fairness_metric",
      col="dataset", height=4, aspect=1.2,
    )
    g.fig.suptitle("Fairness by dataset and model", y=1.02)
    save_fig(g.fig, fig_dir, "09_fairness_by_dataset.png")

  # 8. Test summary
  test_sum = data.get("test_summary", pd.DataFrame())
  if not test_sum.empty:
    cls_metrics = [c for c in ["accuracy", "f1", "roc_auc"] if c in test_sum.columns]
    reg_metrics = [c for c in ["mae", "rmse", "r2"] if c in test_sum.columns]
    use_metrics = [c for c in cls_metrics + reg_metrics if test_sum[c].notna().any()]
    long = test_sum.melt(
      id_vars=["model_label", "dataset", "model_short"],
      value_vars=use_metrics, var_name="metric", value_name="value",
    )
    g = sns.catplot(
      data=long, kind="bar", x="model_short", y="value", col="metric",
      hue="dataset", col_wrap=3, height=4, aspect=1.1, sharey=False,
    )
    g.fig.suptitle("Held-out test performance", y=1.02)
    save_fig(g.fig, fig_dir, "10_test_summary.png")

  print(f"\nAll figures saved under: {fig_dir.resolve()}")
  return fig_dir


In [2]:
for run_key in ACTIVE_RUNS:
  if run_key not in CONFIGS:
    raise KeyError(f"Unknown run key '{run_key}'. Choose from: {list(CONFIGS)}")
  cfg = {**CONFIGS[run_key], "run_key": run_key}
  generate_all_figures(cfg)



Generating figures: Random Forest
  Excel: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\logs\RF Results\random_forest_results.xlsx
  Output: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel


  split_info: (3, 5) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  train_test_distribution: (2, 4) — models: ['RADAR RFC']
  cv_folds: (15, 10) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  cv_balance: (10, 7) — models: ['RADAR RFR', 'RADAR RFC']
  cv_summary: (3, 17) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  subgroup_gender: (6, 8) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  subgroup_age: (9, 8) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  subgroup_gender_age: (18, 8) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  fairness: (3, 3) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']
  test_summary: (3, 11) — models: ['RADAR RFR', 'RADAR RFC', 'Androids RFC']


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\01_split_info.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\02_train_test_distribution_by_dataset.png


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\03_cv_folds_lineplots.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\03_cv_folds_heatmap_accuracy.png


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\03_cv_folds_heatmap_f1.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\03_cv_folds_heatmap_roc_auc.png


...

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\08_subgroup_gender_age_f1.png


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\08_subgroup_gender_age_roc_auc.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\08_subgroup_gender_age_accuracy_heatmap.png


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\09_fairness_combined.png


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\09_fairness_by_dataset.png


Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel\10_test_summary.png

All figures saved under: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\rf_from_excel
